# 04. Qwen2.5 vs Llama-3.1 Tokenizer Subword Fertility Analysis

**Requires GPU + gated-repo access for Llama.** This notebook downloads 8B-parameter models. **Qwen/Qwen2.5-7B-Instruct** is fully open (no authentication needed); **meta-llama/Llama-3.1-8B-Instruct** is gated on HuggingFace Hub -- before running this notebook:
1. Visit https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct while logged in and request/accept access.
2. Authenticate this environment: run `huggingface-cli login` in a terminal/shell cell, or set the `HF_TOKEN` environment variable to a token from https://huggingface.co/settings/tokens.

Run on Colab with a GPU runtime -- see the setup cell below, which auto-clones the repo when a Colab GPU is detected.

In [ ]:
# ============================================================
# PATH & REPO AUTO-SYNC BOOSTER — Guarantees latest project code
# ============================================================
import os, sys, site, urllib.request, zipfile

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)

try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)

home     = os.path.expanduser('~')
proj_dir = os.path.join(home, 'Ekegusii-LLM-Translation-main')
qwen_cfg = os.path.join(proj_dir, 'configs', 'models', 'qwen_7b.yaml')

# Auto-sync if folder is missing OR outdated (lacks qwen_7b.yaml)
if not os.path.isfile(qwen_cfg):
    print('🔄 Outdated or missing repository detected. Auto-syncing latest code from GitHub...')
    zip_path = os.path.join(home, 'repo.zip')
    urllib.request.urlretrieve('https://github.com/aykahsay/Ekegusii-LLM-Translation/archive/refs/heads/main.zip', zip_path)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(home)
    os.remove(zip_path)
    print('✅ Repository auto-synced to latest main commit!')

if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [ ]:
# ============================================================
# ABI CHECK -- numpy/pandas binary compatibility.
# Some Jupyter hosts (e.g. Kineses Cloud conda envs) ship a numpy/pandas
# pair whose compiled C-extension ABI doesn't match, raising
# "numpy.dtype size changed, may indicate binary incompatibility" the
# moment pandas -- and therefore anything importing it, like
# src.master_corpus -- is loaded. Detect and fix it BEFORE any pandas
# import below (see notebooks/00_setup_environment.ipynb for the
# original version of this check).
# ============================================================
import subprocess
import sys


def _abi_ok():
    try:
        import numpy  # noqa: F401
        import pandas  # noqa: F401
        return True
    except ValueError as exc:
        if "binary incompatibility" in str(exc):
            return False
        raise


if not _abi_ok():
    print("numpy/pandas ABI mismatch detected -- attempting fix...")
    fix_a = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "numpy>=2.0.0"],
        capture_output=True, text=True,
    )
    if fix_a.returncode != 0:
        print("  numpy upgrade failed (read-only env?) -- downgrading pandas instead...")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", "pandas==2.2.3"],
            capture_output=True, text=True,
        )
    raise RuntimeError(
        "Fixed numpy/pandas ABI mismatch via pip -- you MUST restart the kernel now "
        "(Kernel -> Restart Kernel) and re-run this notebook from the top. The fix "
        "cannot take effect in the current running process."
    )
else:
    print("numpy/pandas ABI OK.")


In [ ]:
from src.master_corpus.manager import MasterCorpusManager
from src.tokenizer.compare import TokenizerComparator
from src.utils.constants import SUPPORTED_LANGUAGES

manager = MasterCorpusManager()
df = manager.load_sentence_corpus().sample(500, random_state=42)
language_sentences = {lang: df[lang].dropna().astype(str).tolist() for lang in SUPPORTED_LANGUAGES}

## Fertility comparison

In [ ]:
comparator = TokenizerComparator()
fertility_df = comparator.compare(language_sentences)
fertility_df

In [ ]:
relative = comparator.compare(language_sentences)
recommendation = comparator.recommend_base_model(fertility_df, target_language='Ekegusii')
print(f'Recommended base model for Ekegusii: {recommendation}')

## Vocabulary coverage comparison

In [ ]:
comparator.compare_vocabulary_coverage(language_sentences)

## Visualization

In [ ]:
from pathlib import Path
from src.visualization.tokenizer import plot_fertility_comparison, plot_vocabulary_coverage

Path('outputs/figures').mkdir(parents=True, exist_ok=True)
plot_fertility_comparison(fertility_df, output_path='outputs/figures/04_fertility.png')
coverage_df = comparator.compare_vocabulary_coverage(language_sentences)
plot_vocabulary_coverage(coverage_df, output_path='outputs/figures/04_vocab_coverage.png')